# Lab 1 — Data Collection (SpaceX REST API & Web Scraping)
**Author:** Hala Hagag  
**IBM Applied Data Science Capstone**

In this notebook we collect the Falcon 9 launch records from two sources:
1. The public SpaceX REST API (`https://api.spacexdata.com/v4/launches/past`)
2. The Falcon 9 Wikipedia page (via BeautifulSoup web scraping)

The objective is to predict whether the first stage of a Falcon 9 rocket will land successfully — a key driver of launch cost.

In [ ]:
import requests, pandas as pd, numpy as np
from bs4 import BeautifulSoup
pd.set_option('display.max_columns', None)

## 1. SpaceX REST API

In [ ]:
# Helper functions to flatten the API response
def getBoosterVersion(data):
    return [requests.get('https://api.spacexdata.com/v4/rockets/' + r).json()['name'] for r in data['rocket']]

def getLaunchSite(data):
    out=[]
    for s in data['launchpad']:
        r=requests.get('https://api.spacexdata.com/v4/launchpads/' + s).json()
        out.append((r['locality'], r['longitude'], r['latitude'], r['name']))
    return out

def getPayloadData(data):
    masses, orbits = [], []
    for load in data['payloads']:
        r=requests.get('https://api.spacexdata.com/v4/payloads/' + load).json()
        masses.append(r['mass_kg']); orbits.append(r['orbit'])
    return masses, orbits

def getCoreData(data):
    block, reused_count, serial, outcome, flights, gridfins, reused, legs, landing_pad = [],[],[],[],[],[],[],[],[]
    for core in data['cores']:
        if core['core'] is not None:
            r=requests.get('https://api.spacexdata.com/v4/cores/' + core['core']).json()
            block.append(r['block']); reused_count.append(r['reuse_count']); serial.append(r['serial'])
        else:
            block.append(None); reused_count.append(None); serial.append(None)
        outcome.append(f"{core['landing_success']} {core['landing_type']}")
        flights.append(core['flight']); gridfins.append(core['gridfins']); reused.append(core['reused'])
        legs.append(core['legs']); landing_pad.append(core['landpad'])
    return block, reused_count, serial, outcome, flights, gridfins, reused, legs, landing_pad

In [ ]:
# In a connected environment, uncomment to fetch live data
# url='https://api.spacexdata.com/v4/launches/past'
# response=requests.get(url).json()
# df=pd.json_normalize(response)
# For reproducibility we load the prepared dataset
df = pd.read_csv('../data/dataset_part_1.csv')
print(df.shape)
df.head()

## 2. Web Scraping — Falcon 9 Wikipedia Page

In [ ]:
static_url='https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922'
# response=requests.get(static_url).text
# soup=BeautifulSoup(response,'html.parser')
# tables=soup.find_all('table','wikitable')
# print('tables found:', len(tables))

In [ ]:
# After parsing tables we would build the launch dictionary and convert to DataFrame.
# See the data folder for the pre-extracted CSV used in subsequent labs.
print('Data collection completed. Rows:', len(df))